# LeetCode 1371 — Find the Longest Substring Containing Vowels in Even Counts bằng Python

Link đề: https://leetcode.com/problems/find-the-longest-substring-containing-vowels-in-even-counts/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def findTheLongestSubstring(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Biết chuyển điều kiện "mỗi nguyên âm xuất hiện **chẵn** lần" thành điều kiện trên **tính chẵn lẻ (parity)** — chỉ cần biết chẵn/lẻ, không cần biết số lượng.
2. Nắm kỹ thuật **prefix + hiệu**: đoạn `s[i..j]` có tính chất X ⇔ trạng thái prefix tại `j+1` "bằng" trạng thái prefix tại `i`.
3. Mã hóa 5 bit chẵn/lẻ thành **bitmask** `0..31` và cập nhật bằng **XOR**.
4. Dùng mẫu "**lưu vị trí xuất hiện đầu tiên** của mỗi trạng thái" để tìm đoạn **dài nhất** trong `O(n)`.
5. Thấy được con đường cải tiến: `O(n³)` → `O(n²)` → `O(n)` với bộ nhớ `O(1)` (32 ô).

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái thường. Trả về **độ dài** của chuỗi con (liên tiếp) **dài nhất** sao cho mỗi nguyên âm `'a', 'e', 'i', 'o', 'u'` đều xuất hiện một số **chẵn** lần trong đó (0 lần cũng là chẵn).

- **Ví dụ 1:** `s = "eleetminicoworoep"` → `13` (chuỗi con `"leetminicowor"`: hai `e`, hai `i`, hai `o`, không có `a`, `u`).
- **Ví dụ 2:** `s = "leetcodeisgreat"` → `5` (`"leetc"` có hai `e`).
- **Ví dụ 3:** `s = "bcbcbc"` → `6` (không có nguyên âm nào ⇒ cả chuỗi hợp lệ).

**Ràng buộc:** `1 <= s.length <= 5·10^5`; `s` chỉ gồm chữ cái tiếng Anh thường.

Lưu ý: với `n = 5·10^5`, mọi thuật toán `O(n²)` (≈ `2,5·10^11` phép) đều quá chậm — cần `O(n)`.

In [1]:
import random
import time

EXAMPLES = [("eleetminicoworoep", 13), ("leetcodeisgreat", 5), ("bcbcbc", 6)]
VOWELS = "aeiou"
for s, want in EXAMPLES:
    print(f"s = {s!r:22} -> mong đợi {want}")

s = 'eleetminicoworoep'    -> mong đợi 13
s = 'leetcodeisgreat'      -> mong đợi 5
s = 'bcbcbc'               -> mong đợi 6


## 3. Kiến thức nền

### 3.1 Chuỗi con, cắt lát và đếm

`s[i:j]` là chuỗi con từ chỉ số `i` tới `j-1`, độ dài `j - i`. `str.count(c)` đếm số lần xuất hiện — tiện nhưng tốn `O(độ dài)` mỗi lần.

In [2]:
s = "leetminicowor"
print(len(s), {v: s.count(v) for v in VOWELS})
print("leetcodeisgreat"[0:5], "-> độ dài", 5 - 0)

13 {'a': 0, 'e': 2, 'i': 2, 'o': 2, 'u': 0}
leetc -> độ dài 5


### 3.2 Chẵn lẻ và phép XOR

Chỉ cần nhớ **chẵn hay lẻ**: thêm một ký tự `v` làm đảo trạng thái của `v` (chẵn → lẻ → chẵn ...). Với một bit, "đảo" chính là `x ^ 1`. Tính chất quan trọng:
- `x ^ x = 0`, `x ^ 0 = x`, XOR giao hoán và kết hợp.
- Vì vậy nếu `P[j]` là XOR tích lũy tới `j` thì XOR của đoạn `(i, j]` là `P[j] ^ P[i]`.

In [3]:
bit = 0
for ch in "eee":
    bit ^= 1
    print(ch, "-> parity của e =", bit, "(lẻ)" if bit else "(chẵn)")
print("5 ^ 5 =", 5 ^ 5, "| 5 ^ 0 =", 5 ^ 0, "| (3 ^ 6) ^ 6 =", (3 ^ 6) ^ 6)

e -> parity của e = 1 (lẻ)
e -> parity của e = 0 (chẵn)
e -> parity của e = 1 (lẻ)
5 ^ 5 = 0 | 5 ^ 0 = 5 | (3 ^ 6) ^ 6 = 3


### 3.3 Bitmask 5 bit

Gán mỗi nguyên âm một bit: `a → bit 0`, `e → bit 1`, `i → bit 2`, `o → bit 3`, `u → bit 4`. Một số `mask` trong `0..31` mô tả **đồng thời** chẵn/lẻ của cả 5 nguyên âm. `mask == 0` ⇔ tất cả đều chẵn. Dùng `1 << k` để lấy bit thứ `k`, `bin(mask)` hoặc `format(mask, "05b")` để in.

In [4]:
BIT = {v: 1 << k for k, v in enumerate(VOWELS)}
print(BIT)
mask = 0
for ch in "leet":
    mask ^= BIT.get(ch, 0)           # phụ âm: XOR với 0, không đổi
    print(ch, format(mask, "05b"), "(bit đọc từ phải: a e i o u)")

{'a': 1, 'e': 2, 'i': 4, 'o': 8, 'u': 16}
l 00000 (bit đọc từ phải: a e i o u)
e 00010 (bit đọc từ phải: a e i o u)
e 00000 (bit đọc từ phải: a e i o u)
t 00000 (bit đọc từ phải: a e i o u)


### 3.4 Prefix (tiền tố) và mẹo "hai prefix bằng nhau"

Định nghĩa `P[0] = 0`, `P[k]` = mask của `s[0:k]`. Khi đó mask của đoạn `s[i:j]` là `P[j] ^ P[i]`. Đoạn hợp lệ ⇔ `P[j] ^ P[i] == 0` ⇔ **`P[j] == P[i]`**. Bài toán trở thành: *tìm hai chỉ số `i < j` có `P[i] == P[j]` và `j - i` lớn nhất.*

In [5]:
def prefix_masks(s):
    P = [0]
    for ch in s:
        P.append(P[-1] ^ BIT.get(ch, 0))
    return P

P = prefix_masks("leetc")
print("P =", P)
print("đoạn s[0:5] hợp lệ?", P[5] == P[0])

P = [0, 0, 2, 0, 0, 0]
đoạn s[0:5] hợp lệ? True


### 3.5 Dict / mảng lưu "lần xuất hiện đầu tiên"

Để `j - i` lớn nhất với `j` cố định, ta muốn `i` **nhỏ nhất** có cùng giá trị ⇒ chỉ cần lưu **chỉ số đầu tiên** của mỗi giá trị. `dict.setdefault(k, v)` chỉ ghi nếu khóa chưa có — đúng ngữ nghĩa "lần đầu". Vì chỉ có 32 mask, một list dài 32 khởi tạo `-1`/`None` còn nhanh hơn dict.

In [6]:
first = {}
for idx, val in enumerate([3, 1, 3, 2, 1, 3]):
    first.setdefault(val, idx)
print("vị trí đầu tiên:", first)
arr = [None] * 32
arr[0] = 0
print(arr[:6])

vị trí đầu tiên: {3: 0, 1: 1, 2: 3}
[0, None, None, None, None, None]


## 4. Ý tưởng nền tảng / trực giác

Tính tay `s = "leetcodeisgreat"` (mask 5 bit, đọc từ trái: `u o i e a`):

| k | ký tự vừa thêm | P[k] (u o i e a) | ghi chú |
|---|---|---|---|
| 0 | — | 00000 | trạng thái rỗng |
| 1 | l | 00000 | = P[0] ⇒ đoạn dài 1 hợp lệ |
| 2 | e | 00010 | e lẻ |
| 3 | e | 00000 | = P[0] ⇒ đoạn `s[0:3]="lee"` dài 3 |
| 4 | t | 00000 | đoạn `"leet"` dài 4 |
| 5 | c | 00000 | đoạn `"leetc"` dài 5 |
| 6 | o | 01000 | o lẻ |
| 7 | d | 01000 | = P[6] ⇒ đoạn `"d"` dài 1 |
| 8 | e | 01010 | |
| 9 | i | 01110 | lặp lại ở k=10,11,12 nhưng chỉ dài ≤ 3 |
| ... | ... | ... | không có mask nào lặp lại với khoảng cách > 5 |

Đáp án 5. Mấu chốt: **mỗi lần mask lặp lại** là một đoạn hợp lệ kết thúc tại đó; muốn dài nhất thì so với **lần đầu** mask đó xuất hiện.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Ngây thơ nhất: thử mọi đoạn, đếm lại — `O(n³)`

Với mọi cặp `(i, j)`, đếm số lần mỗi nguyên âm trong `s[i:j]`. Đếm lại tốn `O(n)` ⇒ tổng `O(n³)`.

In [7]:
def longest_naive(s):
    n, best = len(s), 0
    for i in range(n):
        for j in range(i + 1, n + 1):
            sub = s[i:j]
            if all(sub.count(v) % 2 == 0 for v in VOWELS):
                best = max(best, j - i)
    return best

for s, want in EXAMPLES:
    print(s, "->", longest_naive(s), "(mong đợi", want, ")")

eleetminicoworoep -> 13 (mong đợi 13 )
leetcodeisgreat -> 5 (mong đợi 5 )
bcbcbc -> 6 (mong đợi 6 )


### 5.2 Baseline tốt hơn: cố định đầu `i`, mở rộng `j` và cập nhật parity — `O(n²)`

Khi kéo dài đoạn thêm một ký tự, chỉ cần đảo 1 bit; không cần đếm lại.

In [8]:
def longest_brute(s):
    n, best = len(s), 0
    for i in range(n):
        mask = 0
        for j in range(i, n):
            mask ^= BIT.get(s[j], 0)
            if mask == 0:
                best = max(best, j - i + 1)
    return best

for s, want in EXAMPLES:
    print(s, "->", longest_brute(s), "(mong đợi", want, ")")

eleetminicoworoep -> 13 (mong đợi 13 )
leetcodeisgreat -> 5 (mong đợi 5 )
bcbcbc -> 6 (mong đợi 6 )


**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** với `n = 5·10^5` cần ≈ `1,25·10^11` bước — không thể. Chúng ta đang xét lại mọi điểm bắt đầu, trong khi chỉ cần biết "điểm bắt đầu **sớm nhất** có cùng prefix-mask". Thử đo thời gian để thấy tăng trưởng bậc hai:

In [9]:
rng = random.Random(1)
for n in [500, 1000, 2000]:
    s = "".join(rng.choice("aeioubcd") for _ in range(n))
    t0 = time.perf_counter()
    longest_brute(s)
    print(f"n={n:5d}: {time.perf_counter() - t0:.3f}s")

n=  500: 0.008s
n= 1000: 0.030s


n= 2000: 0.125s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đoạn `s[i:j]` hợp lệ ⇔ `P[i] == P[j]` (mục 3.4). Vậy:

> Duyệt `j` từ trái sang phải, giữ `first[m]` = chỉ số `k` nhỏ nhất đã thấy với `P[k] = m`. Tại mỗi `j`, ứng viên là `j - first[P[j]]`.

**Quy tắc "nếu… thì…":**
- Nếu mask hiện tại **chưa từng thấy** ⇒ ghi `first[mask] = j` (không cập nhật đáp án).
- Nếu mask **đã thấy** ⇒ cập nhật `best = max(best, j - first[mask])`; **không** ghi đè `first[mask]` (giữ chỉ số nhỏ nhất).
- **Khởi tạo** `first[0] = 0` (theo chỉ số prefix) — tương đương "chỉ số ký tự `-1`" — để các đoạn bắt đầu từ đầu chuỗi được tính.

**Invariant:** sau khi xử lý `j`, `first[m]` đúng bằng `min{k ≤ j : P[k] = m}` (hoặc chưa có), và `best` = độ dài đoạn hợp lệ dài nhất kết thúc tại vị trí `≤ j`.

Vì chỉ có `2^5 = 32` mask, bộ nhớ là hằng số.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Prefix đếm từng nguyên âm (tuple parity) + dict

Trực quan nhất: giữ bộ 5 số chẵn/lẻ dạng tuple làm khóa dict. Chưa cần bitmask.

In [10]:
def longest_tuple(s):
    parity = [0] * 5
    first = {tuple(parity): -1}        # chỉ số ký tự -1 = trước chuỗi
    best = 0
    for j, ch in enumerate(s):
        k = VOWELS.find(ch)
        if k != -1:
            parity[k] ^= 1
        key = tuple(parity)
        if key in first:
            best = max(best, j - first[key])
        else:
            first[key] = j
    return best

for s, want in EXAMPLES:
    print(s, "->", longest_tuple(s), "(mong đợi", want, ")")

eleetminicoworoep -> 13 (mong đợi 13 )
leetcodeisgreat -> 5 (mong đợi 5 )
bcbcbc -> 6 (mong đợi 6 )


Code nộp LeetCode (Cách 2):

In [11]:
class Solution:
    def findTheLongestSubstring(self, s: str) -> int:
        idx = {"a": 0, "e": 1, "i": 2, "o": 3, "u": 4}
        parity = [0] * 5
        first = {(0, 0, 0, 0, 0): -1}
        best = 0
        for j, ch in enumerate(s):
            if ch in idx:
                parity[idx[ch]] ^= 1
            key = tuple(parity)
            if key in first:
                best = max(best, j - first[key])
            else:
                first[key] = j
        return best

print([Solution().findTheLongestSubstring(s) for s, _ in EXAMPLES])

[13, 5, 6]


**Độ phức tạp Cách 2:** `O(n)` lượt, mỗi lượt tạo tuple 5 phần tử (hằng số) ⇒ `O(n)`; dict có tối đa 32 khóa ⇒ `O(1)` bộ nhớ. Chạy đúng nhưng chậm hơn Cách 3 vài lần do tạo tuple.

### 7.2 Cách 3 — Bitmask 5 bit + mảng 32 ô (tối ưu, khuyên dùng)

**Từng đoạn:**
1. `first = [-2] * 32` (giá trị `-2` nghĩa là "chưa thấy"), `first[0] = -1`.
2. Với mỗi `j`, `mask ^= BIT[ch]` nếu `ch` là nguyên âm.
3. Nếu `first[mask] == -2` ⇒ ghi `j`; ngược lại cập nhật `best`.

In [12]:
def longest_bitmask(s):
    first = [-2] * 32
    first[0] = -1
    mask = best = 0
    for j, ch in enumerate(s):
        b = BIT.get(ch)
        if b:
            mask ^= b
        if first[mask] == -2:
            first[mask] = j
        else:
            best = max(best, j - first[mask])
    return best

for s, want in EXAMPLES:
    print(s, "->", longest_bitmask(s), "(mong đợi", want, ")")

eleetminicoworoep -> 13 (mong đợi 13 )
leetcodeisgreat -> 5 (mong đợi 5 )
bcbcbc -> 6 (mong đợi 6 )


Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def findTheLongestSubstring(self, s: str) -> int:
        bit = {"a": 1, "e": 2, "i": 4, "o": 8, "u": 16}
        first = [-2] * 32
        first[0] = -1
        mask = best = 0
        for j, ch in enumerate(s):
            if ch in bit:
                mask ^= bit[ch]
            if first[mask] == -2:
                first[mask] = j
            elif j - first[mask] > best:
                best = j - first[mask]
        return best

print([Solution().findTheLongestSubstring(s) for s, _ in EXAMPLES])

[13, 5, 6]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(32) = O(1)` bộ nhớ.

### 7.3 Cách 4 — Lưu cả lần đầu và lần cuối của mỗi mask

Cách nhìn "tĩnh": tính hết `P[0..n]`, ghi `first[m]` và `last[m]`; đáp án `max(last[m] - first[m])`. Cùng độ phức tạp, nhưng giúp thấy rõ đáp án chỉ phụ thuộc vào **hai đầu mút** của mỗi nhóm prefix bằng nhau.

In [14]:
def longest_first_last(s):
    first = [None] * 32
    last = [None] * 32
    mask = 0
    first[0] = last[0] = 0               # chỉ số prefix
    for k, ch in enumerate(s, start=1):
        mask ^= BIT.get(ch, 0)
        if first[mask] is None:
            first[mask] = k
        last[mask] = k
    return max(last[m] - first[m] for m in range(32) if first[m] is not None)

for s, want in EXAMPLES:
    print(s, "->", longest_first_last(s), "(mong đợi", want, ")")

eleetminicoworoep -> 13 (mong đợi 13 )
leetcodeisgreat -> 5 (mong đợi 5 )
bcbcbc -> 6 (mong đợi 6 )


Code nộp LeetCode (Cách 4):

In [15]:
class Solution:
    def findTheLongestSubstring(self, s: str) -> int:
        bit = {"a": 1, "e": 2, "i": 4, "o": 8, "u": 16}
        first = [-1] * 32
        last = [-1] * 32
        first[0] = last[0] = 0
        mask = 0
        for k, ch in enumerate(s, 1):
            mask ^= bit.get(ch, 0)
            if first[mask] == -1:
                first[mask] = k
            last[mask] = k
        return max(last[m] - first[m] for m in range(32) if first[m] != -1)

print([Solution().findTheLongestSubstring(s) for s, _ in EXAMPLES])

[13, 5, 6]


**Độ phức tạp Cách 4:** `O(n + 32)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace Cách 3 trên `s = "eleetminicoworoep"`. Mỗi dòng in: chỉ số `j`, ký tự, `mask` dạng 5 bit (thứ tự `u o i e a`), `first[mask]`, độ dài ứng viên và `best`.

In [16]:
def trace_bitmask(s):
    first = [-2] * 32
    first[0] = -1
    mask = best = 0
    print(" j | ch | mask  | first[mask] | ứng viên | best")
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        if first[mask] == -2:
            first[mask] = j
            note, cand = "mới -> ghi", "-"
        else:
            cand = j - first[mask]
            best = max(best, cand)
            note = f"{first[mask]:>3}"
        print(f"{j:2d} | {ch}  | {mask:05b} | {note:>11} | {str(cand):>8} | {best}")
    return best

print("kết quả:", trace_bitmask("eleetminicoworoep"))

 j | ch | mask  | first[mask] | ứng viên | best
 0 | e  | 00010 |  mới -> ghi |        - | 0
 1 | l  | 00010 |           0 |        1 | 1
 2 | e  | 00000 |          -1 |        3 | 3
 3 | e  | 00010 |           0 |        3 | 3
 4 | t  | 00010 |           0 |        4 | 4
 5 | m  | 00010 |           0 |        5 | 5
 6 | i  | 00110 |  mới -> ghi |        - | 5
 7 | n  | 00110 |           6 |        1 | 5
 8 | i  | 00010 |           0 |        8 | 8
 9 | c  | 00010 |           0 |        9 | 9
10 | o  | 01010 |  mới -> ghi |        - | 9
11 | w  | 01010 |          10 |        1 | 9
12 | o  | 00010 |           0 |       12 | 12
13 | r  | 00010 |           0 |       13 | 13
14 | o  | 01010 |          10 |        4 | 13
15 | e  | 01000 |  mới -> ghi |        - | 13
16 | p  | 01000 |          15 |        1 | 13
kết quả: 13


Đọc bảng: tại `j = 13` (ký tự `r`), mask quay về `00010` — lần đầu thấy ở `j = 0` (sau chữ `e` đầu). Đoạn `s[1..13] = "leetminicowor"`, độ dài `13 - 0 = 13`. Đó là đáp án.

Bảng tóm tắt các mốc quan trọng:

| j | ký tự | mask | first[mask] | ứng viên |
|---|---|---|---|---|
| 0 | e | 00010 | -1 → ghi 0 | – |
| 2 | e | 00000 | -1 | 3 |
| 3 | e | 00010 | 0 | 3 |
| 13 | r | 00010 | 0 | **13** |
| 12 | o | 00010 | 0 | 12 |
| 16 | p | 01000 | 15 | 1 |

## 9. Code nộp LeetCode

In [17]:
class Solution:
    def findTheLongestSubstring(self, s: str) -> int:
        bit = {"a": 1, "e": 2, "i": 4, "o": 8, "u": 16}
        first = [-2] * 32          # -2: mask chưa xuất hiện
        first[0] = -1              # prefix rỗng kết thúc "trước" chỉ số 0
        mask = best = 0
        for j, ch in enumerate(s):
            if ch in bit:
                mask ^= bit[ch]
            if first[mask] == -2:
                first[mask] = j
            elif j - first[mask] > best:
                best = j - first[mask]
        return best

for s, want in EXAMPLES:
    got = Solution().findTheLongestSubstring(s)
    print(s, got, "OK" if got == want else "SAI")

eleetminicoworoep 13 OK
leetcodeisgreat 5 OK
bcbcbc 6 OK


## 10. Vì sao thuật toán đúng

1. **Tương đương parity:** một nguyên âm xuất hiện chẵn lần trong `s[i:j]` ⇔ số lần trong `s[0:j]` và `s[0:i]` **cùng tính chẵn lẻ**. Áp dụng cho cả 5 nguyên âm ⇔ `P[i] == P[j]`.
2. **Mọi đoạn hợp lệ được xét:** đoạn hợp lệ dài nhất `s[i*:j*]` có `P[i*] = P[j*] = m`. Khi thuật toán xử lý `j*`, `first[m] ≤ i*` (vì `i*` đã được duyệt trước và `first` giữ chỉ số nhỏ nhất) ⇒ ứng viên `j* - first[m] ≥ j* - i*`.
3. **Mọi ứng viên đều hợp lệ:** ứng viên `j - first[m]` ứng với đoạn có hai prefix bằng nhau ⇒ hợp lệ theo (1). Vậy `best` không vượt đáp án thật.

Từ (2) và (3), `best` đúng bằng đáp án. Khởi tạo `first[0]` (prefix rỗng) đảm bảo (2) cũng đúng khi `i* = 0`.

## 11. Phân tích Time / Space

Gọi `n = len(s)`, `V = 5` số nguyên âm (số mask `2^V = 32`).

| Cách | Time | Space |
|---|---|---|
| 1a. Đếm lại mọi đoạn | `O(n³)` (hoặc `O(V·n³)`) | `O(n)` cho chuỗi con |
| 1b. Mở rộng + parity | `O(n²)` | `O(1)` |
| 2. Tuple + dict | `O(V·n)` | `O(2^V)` |
| 3. Bitmask + mảng 32 | `O(n)` | `O(2^V) = O(1)` |
| 4. First/last | `O(n + 2^V)` | `O(2^V)` |

Không có khác biệt average/worst cho Cách 2–4: luôn đúng một lượt qua chuỗi. Với `n = 5·10^5`, Cách 3 chạy cỡ 0,1–0,2 giây trong Python.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "naive": longest_naive,
    "brute": longest_brute,
    "tuple": longest_tuple,
    "bitmask": longest_bitmask,
    "first_last": longest_first_last,
    "submit": lambda s: Solution().findTheLongestSubstring(s),
}
edge = [
    ("a", 0),            # một nguyên âm: không đoạn nào hợp lệ
    ("b", 1),            # một phụ âm
    ("aa", 2),
    ("ae", 0),           # tổng chẵn nhưng từng nguyên âm lẻ
    ("aeiouaeiou", 10),
    ("abca", 4),
    ("uxu", 3),
    ("aab", 3),
    ("baa", 3),
    ("aebe", 3),         # đoạn "ebe" (hai e)
]
for name, fn in METHODS.items():
    for s, want in EXAMPLES + edge:
        got = fn(s)
        assert got == want, (name, s, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 6 cách đúng trên 13 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

In [19]:
rng = random.Random(1371)
for _ in range(500):
    n = rng.randint(1, 40)
    alphabet = rng.choice(["aeioub", "ae", "aeiouxyz", "abcdefghijklmnopqrstuvwxyz"])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    res = {name: fn(s) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, (s, res)
print("500 chuỗi ngẫu nhiên: mọi cách cho cùng kết quả")

500 chuỗi ngẫu nhiên: mọi cách cho cùng kết quả


### 12.3 Test lớn theo ràng buộc (`n = 5·10^5`) — chỉ chạy các cách `O(n)`

In [20]:
rng = random.Random(7)
big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(500_000))
for name in ["tuple", "bitmask", "first_last", "submit"]:
    t0 = time.perf_counter()
    r = METHODS[name](big)
    print(f"{name:10s} -> {r} | {time.perf_counter() - t0:.3f}s")
print("chuỗi toàn phụ âm:", Solution().findTheLongestSubstring("b" * 500_000))

tuple      -> 499978 | 0.086s
bitmask    -> 499978 | 0.056s
first_last -> 499978 | 0.038s
submit     -> 499978 | 0.047s
chuỗi toàn phụ âm: 500000


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên khởi tạo trạng thái rỗng (`first[0] = -1`)

In [21]:
def wrong_no_init(s):
    first = [-2] * 32                 # SAI: không có first[0] = -1
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        if first[mask] == -2:
            first[mask] = j
        else:
            best = max(best, j - first[mask])
    return best

for s in ["bcbcbc", "leetcodeisgreat"]:
    print(s, "-> sai:", wrong_no_init(s), "| đúng:", longest_bitmask(s))

bcbcbc -> sai: 5 | đúng: 6
leetcodeisgreat -> sai: 4 | đúng: 5


**Vì sao sai:** các đoạn **bắt đầu từ chỉ số 0** cần so với prefix rỗng `P[0] = 0`. Không khởi tạo thì lần đầu mask 0 xuất hiện (tại `j = 0`) bị coi là "mới" ⇒ đoạn dài nhất bắt đầu từ đầu chuỗi bị **mất 1 ký tự** (`"bcbcbc"` cho 5 thay vì 6; `"leetcodeisgreat"` cho 4 thay vì 5).

**Code đúng:** `first[0] = -1` (theo chỉ số ký tự) hoặc `first[0] = 0` (theo chỉ số prefix).

### Lỗi 2 — Ghi đè `first[mask]` mỗi lần (lưu lần **cuối** thay vì lần đầu)

In [22]:
def wrong_overwrite(s):
    first = [-2] * 32
    first[0] = -1
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        if first[mask] != -2:
            best = max(best, j - first[mask])
        first[mask] = j               # SAI: luôn ghi đè
    return best

for s in ["eleetminicoworoep", "bcbcbc"]:
    print(s, "-> sai:", wrong_overwrite(s), "| đúng:", longest_bitmask(s))

eleetminicoworoep -> sai: 3 | đúng: 13
bcbcbc -> sai: 1 | đúng: 6


**Vì sao sai:** khi ghi đè, ta chỉ so với **lần gần nhất** mask xuất hiện ⇒ chỉ tìm được đoạn hợp lệ **ngắn nhất** kết thúc tại `j`, không phải dài nhất. `"bcbcbc"` chỉ cho 1.

**Code đúng:** chỉ ghi khi `first[mask]` chưa có giá trị.

### Lỗi 3 — Kiểm tra "tổng số nguyên âm chẵn" thay vì "từng nguyên âm chẵn"

In [23]:
def wrong_total_parity(s):
    first = {0: -1}
    par = best = 0
    for j, ch in enumerate(s):
        if ch in VOWELS:
            par ^= 1                  # SAI: gộp chung một bit cho cả 5 nguyên âm
        if par in first:
            best = max(best, j - first[par])
        else:
            first[par] = j
    return best

for s in ["ae", "leetcodeisgreat"]:
    print(s, "-> sai:", wrong_total_parity(s), "| đúng:", longest_bitmask(s))

ae -> sai: 2 | đúng: 0
leetcodeisgreat -> sai: 13 | đúng: 5


**Vì sao sai:** `"ae"` có tổng 2 nguyên âm (chẵn) nhưng `a` và `e` đều xuất hiện 1 lần (lẻ). Một bit chung không phân biệt được ⇒ trả về 2 thay vì 0.

**Code đúng:** mỗi nguyên âm **một bit riêng** (5 bit).

### Lỗi 4 — Cộng thêm 1 khi tính độ dài (lệch một)

In [24]:
def wrong_plus_one(s):
    first = [-2] * 32
    first[0] = -1
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        if first[mask] == -2:
            first[mask] = j
        else:
            best = max(best, j - first[mask] + 1)   # SAI: +1
    return best

for s in ["bcbcbc", "aa"]:
    print(s, "-> sai:", wrong_plus_one(s), "| đúng:", longest_bitmask(s))

bcbcbc -> sai: 7 | đúng: 6
aa -> sai: 3 | đúng: 2


**Vì sao sai:** `first[mask]` là chỉ số ký tự **cuối của prefix trước**, nên đoạn hợp lệ là `(first[mask], j]`, độ dài đúng `j - first[mask]`. Cộng thêm 1 cho `"bcbcbc"` ra 7 > độ dài chuỗi!

**Code đúng:** `j - first[mask]` (khi `first` lưu chỉ số ký tự) hoặc `k - first[mask]` (khi lưu chỉ số prefix) — chọn **một** quy ước và giữ nhất quán.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "**mỗi loại xuất hiện chẵn lần**" / "**số lẻ không quá 1**" ⇒ nghĩ ngay **bitmask parity + XOR prefix**.
- Thấy "**đoạn con dài nhất có tổng/trạng thái = 0**" ⇒ prefix + dict **lần xuất hiện đầu tiên**. Nếu đề hỏi **đếm số đoạn** ⇒ prefix + **Counter** số lần xuất hiện.
- Số trạng thái nhỏ (≤ 2^10) ⇒ dùng **list** thay dict cho nhanh.
- Ký tự không liên quan (phụ âm) = XOR với 0 ⇒ chỉ cần không làm gì.
- Luôn tự hỏi: "trạng thái rỗng đã được đưa vào chưa?" — lỗi phổ biến nhất của dạng prefix.

## 15. Biến thể của bài

1. **Đếm số chuỗi con** có mọi nguyên âm chẵn: thay "first" bằng `cnt[mask]`, cộng `cnt[mask]` trước khi tăng.
2. **LeetCode 1542 — Longest Awesome Substring:** chữ số `0..9`, đoạn có thể hoán vị thành palindrome ⇔ mask có **≤ 1 bit** bật ⇒ thử `mask` và `mask ^ (1 << d)` cho 10 chữ số.
3. **LeetCode 1915 — Number of Wonderful Substrings:** đếm đoạn có ≤ 1 ký tự lẻ (10 chữ cái `a..j`).
4. **LeetCode 525 — Contiguous Array:** dãy 0/1, đoạn có số 0 bằng số 1 ⇒ prefix tổng (+1/−1) và lần đầu.
5. **Mọi nguyên âm xuất hiện lẻ lần:** đoạn hợp lệ ⇔ `P[i] ^ P[j] == 31` ⇔ `P[i] == P[j] ^ 31`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Ngây thơ | thử mọi đoạn, `count` lại | `O(n³)` | `O(n)` | chỉ để kiểm chứng, `n ≤ 100` |
| 1b. Mở rộng | cố định đầu, cập nhật mask | `O(n²)` | `O(1)` | brute force tham chiếu, `n ≤ 5000` |
| 2. Tuple + dict | prefix parity dạng tuple, lần đầu | `O(5n)` | `O(32)` | khi chưa quen bitmask |
| 3. Bitmask + mảng | prefix XOR 5 bit, `first[32]` | `O(n)` | `O(1)` | **lời giải chuẩn** |
| 4. First/last | đáp án = max(last − first) mỗi mask | `O(n)` | `O(1)` | khi cần thông tin cả hai đầu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `findTheLongestSubstring` trả về gì với `"aeiouaeiou"`, `"aeiou"`, `"xaxexix"`?

**Bài 2 (tính tay):** liệt kê `P[0..6]` (dạng số nguyên) cho `s = "banana"`.

**Bài 3 (điền TODO):** đếm **số** chuỗi con có mọi nguyên âm xuất hiện chẵn lần.

In [25]:
def count_even_vowel_substrings(s):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    cnt = [0] * 32
    # TODO: trạng thái rỗng xuất hiện 1 lần
    mask = total = 0
    for ch in s:
        # TODO: cập nhật mask; cộng cnt[mask] vào total; tăng cnt[mask]
        pass
    return total

**Bài 4 (tự viết lại):** viết lại Cách 3 nhưng dùng `dict` và `setdefault`, không dùng mảng 32 ô.

**Bài 5 (mini challenge):** độ dài chuỗi con dài nhất mà **nhiều nhất một** nguyên âm xuất hiện lẻ lần (các nguyên âm khác chẵn).

In [26]:
def longest_at_most_one_odd(s):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Đếm từng nguyên âm của cả chuỗi trước | `"aeiou"`: không có hai prefix bằng nhau ngoài... | 10, 0, 1 (`"x"`) |
| 2 | Chỉ `a` đổi mask (bit 0) | mask bật/tắt luân phiên tại mỗi `a` | `[0,0,1,1,0,0,1]` |
| 3 | Thay "lần đầu" bằng "số lần" | `cnt[0] = 1` trước vòng lặp | `total += cnt[mask]; cnt[mask] += 1` |
| 4 | `first = {0: -1}` | `first.setdefault(mask, j)` trả về giá trị đang lưu | `best = max(best, j - first.setdefault(mask, j))` |
| 5 | Đoạn hợp lệ ⇔ `P[i] ^ P[j]` có ≤ 1 bit | thử `m = mask` và `m = mask ^ (1<<k)` với 5 bit | lấy max `j - first[m]` trên 6 ứng viên |

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = [longest_bitmask(x) for x in ["aeiouaeiou", "aeiou", "xaxexix"]]
print("Bài 1:", ans1)
assert ans1 == [10, 0, 1]

# Bài 2
ans2 = prefix_masks("banana")
print("Bài 2:", ans2)
assert ans2 == [0, 0, 1, 1, 0, 0, 1]

Bài 1: [10, 0, 1]
Bài 2: [0, 0, 1, 1, 0, 0, 1]


In [28]:
# Bài 3
def count_even_vowel_substrings_sol(s):
    cnt = [0] * 32
    cnt[0] = 1
    mask = total = 0
    for ch in s:
        mask ^= BIT.get(ch, 0)
        total += cnt[mask]
        cnt[mask] += 1
    return total

def count_brute(s):
    return sum(1 for i in range(len(s)) for j in range(i + 1, len(s) + 1)
               if all(s[i:j].count(v) % 2 == 0 for v in VOWELS))

assert count_even_vowel_substrings_sol("bcbcbc") == 21      # mọi đoạn: 6*7/2
assert count_even_vowel_substrings_sol("aa") == 1
rng = random.Random(3)
for _ in range(200):
    s = "".join(rng.choice("aeb") for _ in range(rng.randint(1, 15)))
    assert count_even_vowel_substrings_sol(s) == count_brute(s), s
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def longest_dict(s):
    first = {0: -1}
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        best = max(best, j - first.setdefault(mask, j))   # mask mới -> j - j = 0
    return best

rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("aeiouxy") for _ in range(rng.randint(1, 30)))
    assert longest_dict(s) == longest_brute(s)
print("Bài 4: OK")

Bài 4: OK


In [30]:
# Bài 5
def longest_at_most_one_odd_sol(s):
    first = [None] * 32
    first[0] = -1
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= BIT.get(ch, 0)
        for m in [mask] + [mask ^ (1 << k) for k in range(5)]:
            if first[m] is not None:
                best = max(best, j - first[m])
        if first[mask] is None:
            first[mask] = j
    return best

def at_most_one_odd_brute(s):
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            odd = sum(s[i:j].count(v) % 2 for v in VOWELS)
            if odd <= 1:
                best = max(best, j - i)
    return best

assert longest_at_most_one_odd_sol("a") == 1
assert longest_at_most_one_odd_sol("ae") == 1
assert longest_at_most_one_odd_sol("aeb") == 2
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("aeiob") for _ in range(rng.randint(1, 18)))
    assert longest_at_most_one_odd_sol(s) == at_most_one_odd_brute(s), s
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đoạn dài nhất có trạng thái prefix bằng nhau":**

```python
first = [-2] * NUM_STATES      # hoặc dict
first[0] = -1                  # TRẠNG THÁI RỖNG!
state = best = 0
for j, ch in enumerate(s):
    state ^= bit_of(ch)        # cập nhật trạng thái prefix
    if first[state] == -2:
        first[state] = j       # chỉ ghi LẦN ĐẦU
    else:
        best = max(best, j - first[state])   # KHÔNG +1
return best
```

**Checklist trước khi submit:**
- [ ] Mỗi nguyên âm một bit riêng (5 bit), phụ âm không đổi mask.
- [ ] Đã khởi tạo trạng thái rỗng `first[0] = -1`.
- [ ] Chỉ ghi `first[mask]` lần đầu, không ghi đè.
- [ ] Độ dài là `j - first[mask]`, không `+1`.
- [ ] Thử `"bcbcbc"` (6), `"a"` (0), `"ae"` (0), `"aa"` (2).
- [ ] Không có vòng lặp lồng nhau — `n = 5·10^5` cần `O(n)`.
- [ ] Đúng chữ ký: `findTheLongestSubstring(self, s: str) -> int`.